# 🦾 J.A.R.V.I.S. HUD — Computer Vision Workshop

Welcome! By the end of this notebook you'll have a **live Iron Man-style HUD** running on your webcam.

### Rules of the road
- Run cells **in order** — Shift+Enter, or click the ▶ button
- Look for `# 🔧 YOUR TURN` — that's the only place you need to type anything
- If something breaks, raise your hand or re-run from the top

| Section | What happens | Time |
|---------|-------------|------|
| 1 | Install libraries | ~30 sec |
| 2 | See yourself on screen | ~1 min |
| 3 | Face detection + HUD | ~2 min |
| 4 | Make it yours | you control this |
| 5 | Bonus: health bar | if you finish early |

---
## 📦 Section 1 — Install & Import
Run this once. It takes about 30 seconds. You'll see `✅ Ready!` when it's done.

In [3]:
!pip install mediapipe opencv-python-headless --quiet

import cv2
import mediapipe as mp
import numpy as np
from IPython.display import display, Javascript, Image as IPImage
from google.colab.output import eval_js
from base64 import b64decode
import time

# ── Webcam helper (no need to read this — just run it) ──────────────────────
def capture_frame():
    js = Javascript('''
        async function capture() {
            const video = document.createElement('video');
            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            video.srcObject = stream;
            await new Promise(r => video.onloadedmetadata = r);
            video.play();
            await new Promise(r => setTimeout(r, 500));
            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getTracks().forEach(t => t.stop());
            return canvas.toDataURL('image/jpeg', 0.8);
        }
        capture().then(result => element.textContent = result);
    ''')
    display(js)
    data_url = eval_js('capture()')
    img_bytes = b64decode(data_url.split(',')[1])
    img_array = np.frombuffer(img_bytes, dtype=np.uint8)
    return cv2.imdecode(img_array, cv2.IMREAD_COLOR)

def show_image(img, title=''):
    _, buffer = cv2.imencode('.jpg', img)
    display(IPImage(data=buffer.tobytes()))
    if title: print(f"👆 {title}")

# Load the face detector
detector = mp.solutions.face_detection.FaceDetection(
    model_selection=0, min_detection_confidence=0.5
)

print("✅ Ready!")

zsh:1: command not found: pip


ModuleNotFoundError: No module named 'cv2'

---
## 📸 Section 2 — Hello Webcam
Run this cell. Your browser will ask for camera permission — click **Allow**.

You should see a photo of yourself appear below the cell.

In [2]:
print("📷 Smile! Capturing...")
frame = capture_frame()
show_image(frame, title=f"Your webcam feed — {frame.shape[1]}x{frame.shape[0]}px")

📷 Smile! Capturing...


NameError: name 'capture_frame' is not defined

---
## 🤖 Section 3 — Face Detection + HUD

This cell does everything at once:
1. Captures a frame from your webcam
2. Runs a pre-trained AI model to find faces
3. Draws the Iron Man HUD on top

Just run it — **no changes needed yet**. Customization is in Section 4!

In [ ]:
# ── HUD Settings (you'll customize these in Section 4) ──────────────────────
HUD_COLOR    = (0, 255, 180)    # color of boxes & text  (B, G, R)
LABEL_BG     = (0, 80, 50)      # label background color (B, G, R)
STATUS_COLOR = (0, 200, 255)    # status bar text color  (B, G, R)
PILOT_NAME   = "PILOT: TONY STARK"
STATUS_MSG   = "JARVIS ACTIVE  |  THREAT LEVEL: LOW  |  SYSTEMS: NOMINAL"

# ── Drawing helpers ──────────────────────────────────────────────────────────
def draw_corners(img, x, y, w, h, color, length=22):
    """Draws the 4 corner brackets that give the sci-fi targeting look."""
    pts = [
        ((x,     y),     (x+length, y),     (x,     y+length)),
        ((x+w,   y),     (x+w-length, y),   (x+w,   y+length)),
        ((x,     y+h),   (x+length, y+h),   (x,     y+h-length)),
        ((x+w,   y+h),   (x+w-length, y+h), (x+w,   y+h-length)),
    ]
    for corner, h_pt, v_pt in pts:
        cv2.line(img, corner, h_pt, color, 2)
        cv2.line(img, corner, v_pt, color, 2)

def draw_label(img, text, x, y, color, bg):
    """Draws a filled label box with text above the bounding box."""
    font = cv2.FONT_HERSHEY_SIMPLEX
    (tw, th), _ = cv2.getTextSize(text, font, 0.55, 1)
    cv2.rectangle(img, (x, y - th - 10), (x + tw + 10, y), bg, -1)
    cv2.putText(img, text, (x + 5, y - 5), font, 0.55, color, 1, cv2.LINE_AA)

def draw_status(img, msg, color):
    """Draws a status bar across the bottom of the frame."""
    h, w = img.shape[:2]
    cv2.rectangle(img, (0, h - 30), (w, h), (0, 0, 0), -1)
    cv2.putText(img, msg, (10, h - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)

# ── Main: capture → detect → draw ───────────────────────────────────────────
print("📷 Capturing frame...")
frame  = capture_frame()
h, w   = frame.shape[:2]
output = frame.copy()

results = detector.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

if results.detections:
    for i, det in enumerate(results.detections):
        score = det.score[0]
        box   = det.location_data.relative_bounding_box
        x  = max(0, int(box.xmin * w))
        y  = max(0, int(box.ymin * h))
        bw = int(box.width  * w)
        bh = int(box.height * h)

        # Subtle tint inside the bounding box
        overlay = output.copy()
        cv2.rectangle(overlay, (x, y), (x+bw, y+bh), HUD_COLOR, -1)
        cv2.addWeighted(overlay, 0.08, output, 0.92, 0, output)

        draw_corners(output, x, y, bw, bh, HUD_COLOR)
        draw_label(output, f"{PILOT_NAME}  |  CONF: {score:.0%}", x, y, HUD_COLOR, LABEL_BG)

    print(f"🎯 {len(results.detections)} face(s) detected — HUD online!")
else:
    print("😅 No faces found — try better lighting or move closer.")

draw_status(output, STATUS_MSG, STATUS_COLOR)
show_image(output, title="J.A.R.V.I.S. HUD")

---
## 🎨 Section 4 — Make It Yours

This is your creative time! Change the values below, then run the cell.

**Colors** are written as `(Blue, Green, Red)` — each number is 0–255.
Some examples to try:

| Color | Value |
|-------|-------|
| Iron Man red | `(0, 0, 220)` |
| Arc reactor blue | `(255, 180, 0)` |
| War Machine silver | `(200, 200, 200)` |
| Rescue gold | `(0, 215, 255)` |
| Default cyan-green | `(0, 255, 180)` |

In [ ]:
# ════════════════════════════════════════════════
#   🔧 YOUR TURN — change anything below!
# ════════════════════════════════════════════════

HUD_COLOR    = (0, 255, 180)       # 🔧 try a different color!
LABEL_BG     = (0, 80, 50)         # 🔧 label background
STATUS_COLOR = (0, 200, 255)       # 🔧 status bar text

PILOT_NAME   = "PILOT: TONY STARK" # 🔧 put YOUR name here!
STATUS_MSG   = "JARVIS ACTIVE  |  THREAT LEVEL: LOW  |  SYSTEMS: NOMINAL"
#                                    🔧 change the status message!

# ════════════════════════════════════════════════
#   Everything below runs automatically — no edits needed
# ════════════════════════════════════════════════
print("📷 Capturing your custom HUD...")
frame  = capture_frame()
h, w   = frame.shape[:2]
output = frame.copy()
results = detector.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

if results.detections:
    for det in results.detections:
        score = det.score[0]
        box   = det.location_data.relative_bounding_box
        x  = max(0, int(box.xmin * w))
        y  = max(0, int(box.ymin * h))
        bw = int(box.width  * w)
        bh = int(box.height * h)
        overlay = output.copy()
        cv2.rectangle(overlay, (x, y), (x+bw, y+bh), HUD_COLOR, -1)
        cv2.addWeighted(overlay, 0.08, output, 0.92, 0, output)
        draw_corners(output, x, y, bw, bh, HUD_COLOR)
        draw_label(output, f"{PILOT_NAME}  |  CONF: {score:.0%}", x, y, HUD_COLOR, LABEL_BG)
    print(f"🎯 {len(results.detections)} face(s) detected!")
else:
    print("😅 No faces found — try again!")

draw_status(output, STATUS_MSG, STATUS_COLOR)
show_image(output, title="Your Custom HUD!")

---
## ⭐ Section 5 — Bonus: Add a Health Bar

For fast finishers! This adds a power bar below each detected face.

Change `POWER_LEVEL` to any number between 0 and 100.

In [ ]:
# 🔧 YOUR TURN: set your suit's power level!
POWER_LEVEL  = 87     # any number 0–100
PILOT_NAME   = "PILOT: TONY STARK"   # 🔧 your name
HUD_COLOR    = (0, 255, 180)          # 🔧 your color
LABEL_BG     = (0, 80, 50)
STATUS_COLOR = (0, 200, 255)

# ── Power bar drawing helper ─────────────────────────────────────────────────
def draw_power_bar(img, x, y, w, level, color):
    bar_y  = y + 6
    fill_w = int(w * (level / 100))
    cv2.rectangle(img, (x, bar_y), (x + w, bar_y + 8), (40, 40, 40), -1)
    cv2.rectangle(img, (x, bar_y), (x + fill_w, bar_y + 8), color, -1)
    cv2.putText(img, f"PWR {level}%", (x + w + 6, bar_y + 8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1, cv2.LINE_AA)

# ── Capture & draw ───────────────────────────────────────────────────────────
print("📷 Capturing...")
frame   = capture_frame()
h, w    = frame.shape[:2]
output  = frame.copy()
results = detector.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

if results.detections:
    for det in results.detections:
        score = det.score[0]
        box   = det.location_data.relative_bounding_box
        x  = max(0, int(box.xmin * w))
        y  = max(0, int(box.ymin * h))
        bw = int(box.width  * w)
        bh = int(box.height * h)
        overlay = output.copy()
        cv2.rectangle(overlay, (x, y), (x+bw, y+bh), HUD_COLOR, -1)
        cv2.addWeighted(overlay, 0.08, output, 0.92, 0, output)
        draw_corners(output, x, y, bw, bh, HUD_COLOR)
        draw_label(output, f"{PILOT_NAME}  |  CONF: {score:.0%}", x, y, HUD_COLOR, LABEL_BG)
        draw_power_bar(output, x, y + bh, bw, POWER_LEVEL, HUD_COLOR)
    print(f"🎯 {len(results.detections)} face(s) detected — suit at {POWER_LEVEL}% power!")
else:
    print("😅 No faces — try again!")

draw_status(output, f"POWER: {POWER_LEVEL}%  |  JARVIS ACTIVE  |  SYSTEMS: NOMINAL", STATUS_COLOR)
show_image(output, title=f"HUD + Power Bar ({POWER_LEVEL}%)")

---
## 🎉 You built it!

| What you did | How it works |
|---|---|
| Captured webcam frames | JavaScript → base64 → numpy array |
| Detected faces | MediaPipe model (trained on millions of images) |
| Drew the HUD | OpenCV rectangles, lines, and text |
| Customized it | Just changing Python variables |

### Want to go further?
- [MediaPipe](https://developers.google.com/mediapipe) — hand tracking, pose, iris detection
- [Ultralytics YOLOv8](https://docs.ultralytics.com) — detect any object (phones, cars, animals)
- [Roboflow](https://roboflow.com) — train a model on your OWN custom photos

---
*Computer Vision Workshop · Built with MediaPipe + OpenCV*